# Showjumping SSL — Final Notebook

## 0. Setup

In [ ]:
import os, sys, subprocess
from pathlib import Path

GITHUB_REPO = 'https://github.com/bballhaus/showjumping-ssl.git'
DRIVE_DATA_ROOT = '/content/drive/MyDrive/CS131'
BRANCH = 'main'

from google.colab import drive
drive.mount('/content/drive')
Path(f'{DRIVE_DATA_ROOT}/data').mkdir(parents=True, exist_ok=True)
Path(f'{DRIVE_DATA_ROOT}/checkpoints').mkdir(parents=True, exist_ok=True)

REPO_DIR = Path('/content/project')
if REPO_DIR.exists():
    subprocess.run(['git', '-C', str(REPO_DIR), 'fetch', '--all'], check=True)
    subprocess.run(['git', '-C', str(REPO_DIR), 'reset', '--hard', f'origin/{BRANCH}'], check=True)
else:
    subprocess.run(['git', 'clone', '--branch', BRANCH, '--depth', '1',
                    GITHUB_REPO, str(REPO_DIR)], check=True)

for sub in ['data', 'checkpoints']:
    target = Path(f'{DRIVE_DATA_ROOT}/{sub}')
    link = REPO_DIR / sub
    if link.is_symlink() or link.exists():
        if link.is_dir() and not link.is_symlink():
            subprocess.run(['rm', '-rf', str(link)], check=True)
        else:
            link.unlink(missing_ok=True)
    link.symlink_to(target, target_is_directory=True)

os.chdir(REPO_DIR)
sys.path.insert(0, str(REPO_DIR))

for _m in [k for k in list(sys.modules) if k == 'src' or k.startswith('src.')]:
    del sys.modules[_m]

print('cwd:', os.getcwd())
print('data ->', os.readlink('data'))
print('checkpoints ->', os.readlink('checkpoints'))

### 0b. Refresh code (fast)

In [ ]:
import subprocess, sys
subprocess.run(['git', '-C', REPO_DIR, 'fetch', '--all', '-q'], check=True)
subprocess.run(['git', '-C', REPO_DIR, 'reset', '--hard', f'origin/{BRANCH}'], check=True)
for _m in [k for k in list(sys.modules) if k == 'src' or k.startswith('src.')]:
    del sys.modules[_m]
head = subprocess.run(['git', '-C', REPO_DIR, 'log', '-1', '--oneline'],
                      capture_output=True, text=True).stdout.strip()
print('refreshed code ->', head)

## 1. Install dependencies

In [ ]:
!pip install -q -r requirements.txt
!pip install -q ultralytics
!apt-get -qq install -y ffmpeg

import torch
print(f'torch: {torch.__version__} | cuda: {torch.cuda.is_available()}')
!nvidia-smi 2>&1 | head -10

## 2. Sanity check — clips present

In [ ]:
from pathlib import Path
from collections import Counter

clips = sorted(Path('data/clips').glob('*.mp4'))
assert clips, 'data/clips is empty. Run colab_milestone.ipynb through clip segmentation first.'
by_v = Counter(p.stem.split('_', 1)[0] for p in clips)
print(f'{len(clips)} clips across {len(by_v)} source videos:')
for v, c in sorted(by_v.items()):
    print(f'  {v}: {c}')

## 3. Cross-venue fence annotation

In [ ]:
!pip install -q jupyter_bbox_widget ipywidgets
from google.colab import output
output.enable_custom_widget_manager()

from src.preprocess.annotate_colab import ColabAnnotator

ann = ColabAnnotator('data/clips', 'data/annotations/fences.csv', limit=120)
ann.start()

## 4. Outcome annotation

### 4a. Rank clips by jump likelihood

In [ ]:
import subprocess
subprocess.run(['python', '-m', 'src.preprocess.jump_filter'],
               check=True, cwd='/content/project')

### 4b. Cut missing jump clips from raw

In [ ]:
from pathlib import Path
import pandas as pd
from src.data.segment import cut_clip

raw_dir, clips_dir = Path('data/raw'), Path('data/clips')
clips_dir.mkdir(parents=True, exist_ok=True)

jumps = (pd.read_csv('data/annotations/jump_candidates.csv')
           .query('is_jump == True')['clip_id'].astype(str).tolist())
on_disk = {p.stem for p in clips_dir.glob('*.mp4')}
missing = [c for c in jumps if c not in on_disk]
print(f'{len(missing)} jump clips to cut')

made, failed = 0, []
for cid in missing:
    vid, ms = cid.rsplit('_', 1)
    src = next(iter(raw_dir.glob(f'{vid}.*')), None)
    if src is None:
        failed.append((cid, 'no raw video')); continue
    if cut_clip(src, int(ms) / 1000, 2.0, clips_dir / f'{cid}.mp4'):
        made += 1
    else:
        failed.append((cid, 'ffmpeg failed'))
print(f'cut {made}, failed {len(failed)}')
for f in failed[:10]:
    print(' ', f)

### 4c. Backfill skipped clips

In [ ]:
import pandas as pd
from pathlib import Path
ann = Path('data/annotations')
cand = pd.read_csv(ann/'jump_candidates.csv')['clip_id'].astype(str).tolist()
labeled = set(pd.read_csv(ann/'outcomes.csv')['clip_id'].astype(str))
skips = ann/'skips.csv'
already = set(skips.read_text().split()) if skips.exists() else set()
ranks = [i for i, c in enumerate(cand) if c in labeled]
depth = max(ranks) + 1 if ranks else 0
backfill = [c for c in cand[:depth] if c not in labeled and c not in already]
with skips.open('a') as f:
    for c in backfill:
        f.write(c + '\n')
print(f'labeled={len(labeled)} depth={depth} backfilled={len(backfill)} '
      f'total_skips={len(already)+len(backfill)}')

### 4d. Label outcomes

In [ ]:
from google.colab import output
output.enable_custom_widget_manager()

import pandas as pd
from src.preprocess.annotate_colab import OutcomeAnnotator

cand = pd.read_csv('data/annotations/jump_candidates.csv')
jumps = cand[cand['is_jump']]['clip_id'].astype(str).tolist()
print(f'{len(jumps)} jump candidates - labeling highest-lift first; Quit when canters appear')
ann = OutcomeAnnotator('data/clips', 'data/annotations/outcomes.csv',
                       limit=len(jumps), only_clips=jumps)
ann.start()

## 5. Fine-tune fence YOLOv8 + vertical/oxer CNN

In [ ]:
from pathlib import Path
from src.preprocess.fence_yolo import export, train as yolo_train
from src.preprocess.fence_type_cnn import train as type_train

yaml_path = export(Path('data/clips'), Path('data/annotations/fences.csv'),
                   Path('data/fence_yolo'), val_frac=0.2)
best = yolo_train(yaml_path, weights='yolov8n.pt', epochs=100, imgsz=320, device=0)
print('fence YOLO best weights:', best)

type_train(Path('data/clips'), Path('data/annotations/fences.csv'),
           Path('checkpoints/fence_type.pt'), epochs=40, device='cuda')

## 6. YOLO horse + geometric d → auto.csv

In [ ]:
from pathlib import Path
import pandas as pd
from tqdm.auto import tqdm
from src.preprocess.run_pipeline import process_clip, load_fence_annotations
from src.preprocess.detect import HorseDetector
from src.preprocess.fence_yolo import FenceDetector

fences = load_fence_annotations(Path('data/annotations/fences.csv'))
clips = sorted(Path('data/clips').glob('*.mp4'))
if not clips:
    print('no clips on disk - skip this cell and run Section 6b (recompute from cached tracks.json)')
else:
    detector = HorseDetector(weights='yolov8n.pt', device='cuda')
    fence_ckpt = Path('checkpoints/fence_yolo/train/weights/best.pt')
    fence_det = FenceDetector(str(fence_ckpt), device='cuda') if fence_ckpt.exists() else None
    auto = 'auto-detect on for the rest' if fence_det else 'no fence YOLO (run Section 5 to enable) - d only on hand-boxed clips'
    print(f'{len(clips)} clips, {len(fences)} hand boxes, {auto}')

    rows = [process_clip(cp, detector, fences.get(cp.stem), fence_detector=fence_det)
            for cp in tqdm(clips, desc='YOLO + geometry', unit='clip')]
    df = pd.DataFrame(rows)
    df.to_csv('data/annotations/auto.csv', index=False)

    pd.DataFrame({'clip_id': [p.stem for p in clips],
                 'video': [p.stem.split('_', 1)[0] for p in clips]}
                ).to_csv('data/annotations/by_video.csv', index=False)

    typed = (df['type'].astype(str).isin(('vertical', 'oxer'))).sum()
    print(f'auto.csv: {len(df)} rows, {typed} typed, {df["d_meters"].notna().sum()} with d')

### 6b. Recompute takeoff + d from cached tracks (no clips / no YOLO)

In [ ]:
from pathlib import Path
import subprocess

REPO = '/content/project'
for f in ['data/annotations/tracks.json',
          'data/annotations/fences.csv',
          'data/annotations/takeoffs.csv']:
    if not Path(f).exists():
        Path(f).parent.mkdir(parents=True, exist_ok=True)
        with open(f, 'wb') as out:
            subprocess.run(['git', '-C', REPO, 'show', f'HEAD:{f}'], stdout=out, check=True)
        print('restored from git:', f)

subprocess.run(['python', '-m', 'src.preprocess.takeoff_local'], check=True, cwd=REPO)

## 7. Merge labels → labels.csv

In [ ]:
from pathlib import Path
from src.data.labels import build_labels

build_labels(Path('data/annotations/auto.csv'),
             Path('data/annotations/outcomes.csv'),
             Path('data/annotations/by_video.csv'),
             Path('data/annotations/labels.csv'))

## 8. SSL pretraining with the venue fix

In [ ]:
import subprocess, os
os.makedirs('data/results', exist_ok=True)
logf = open('data/results/ssl_train.log', 'w')
proc = subprocess.Popen(
    ['python', '-m', 'src.ssl.train',
     '--clips', 'data/clips', '--out', 'checkpoints',
     '--epochs', '15', '--batch-size', '32', '--workers', '2',
     '--lambda-order', '0.3', '--lambda-domain', '0.3',
     '--kinetics-init', '--balance-venues', '--device', 'cuda'],
    stdout=logf, stderr=subprocess.STDOUT, cwd='/content/project')
print('SSL training PID', proc.pid, '- watch: !tail -n 20 data/results/ssl_train.log')

In [ ]:
!tail -n 20 data/results/ssl_train.log

## 9. Embeddings

In [ ]:
from pathlib import Path
from src.ssl.embed import embed_dir

embed_dir(Path('data/clips'), Path('checkpoints/encoder.pt'),
          Path('data/embeddings.npz'), device='cuda', batch_size=16)

## 10. Downstream heads (frozen vs. fine-tuned, cross-venue split)

In [ ]:
from pathlib import Path
from src.downstream.train import run

labels = Path('data/annotations/labels.csv')
clips = Path('data/clips')

print('== frozen SSL encoder ==')
print(run(labels, clips, ckpt=Path('checkpoints/encoder.pt'), finetune=False,
          task='both', group_by_venue=True, epochs=30, device='cuda'))

print('== fine-tuned SSL encoder ==')
print(run(labels, clips, ckpt=Path('checkpoints/encoder.pt'), finetune=True,
          task='both', group_by_venue=True, epochs=30, device='cuda'))

## 11. Baselines

In [ ]:
from pathlib import Path
from src.downstream.train import run
from src.baselines.imagenet2d import ImageNet2DEncoder
from src.baselines.type_logreg import run as logreg_run

labels = Path('data/annotations/labels.csv')
clips = Path('data/clips')

print('type_logreg :', logreg_run(labels, group_by_venue=True))
print('imagenet2d  :', run(labels, clips, encoder=ImageNet2DEncoder(), finetune=False,
                           task='both', group_by_venue=True, epochs=30, device='cuda'))
print('from_scratch:', run(labels, clips, ckpt=None, kinetics_init=False, finetune=True,
                          task='both', group_by_venue=True, epochs=40, device='cuda'))

## 12. Sample-efficiency sweep

In [ ]:
from pathlib import Path
from src.eval.sample_efficiency import sweep

sweep(Path('data/annotations/labels.csv'), Path('data/clips'),
      Path('data/results/sample_efficiency.csv'),
      ckpt=Path('checkpoints/encoder.pt'),
      ns=(25, 50, 100, 200), seeds=(0, 1, 2),
      group_by_venue=True, epochs=30, device='cuda')

### 12b. Honest cross-venue sample efficiency (held-out venue)

In [ ]:
import pandas as pd
from src.eval.sample_efficiency import sweep
from src.eval.cross_venue import _video_id
from src.data.venues import venue_of

lab = pd.read_csv('data/annotations/labels.csv')
lab = lab[lab['outcome'].isin(['clean','knockdown','refusal'])].copy()
lab['venue'] = lab['clip_id'].map(lambda c: venue_of(_video_id(c)))
print(pd.crosstab(lab['venue'], lab['outcome']))

singles = [v for v in ('tryon','madrid') if (lab['venue']==v).any()]
held = max(singles, key=lambda v: (lab['venue']==v).sum())
print('held-out venue =', held)

sweep(Path('data/annotations/labels.csv'), Path('data/clips'),
      Path('data/results/sample_efficiency_venue.csv'),
      ckpt=Path('checkpoints/encoder.pt'),
      ns=(25, 50, 100), seeds=(0, 1, 2),
      val_venue=held, epochs=30, device='cuda')

from src.viz.sample_efficiency import plot as plot_se
plot_se(Path('data/results/sample_efficiency_venue.csv'),
        Path('milestone/figures/sample_efficiency_venue_f1.png'), metric='macro_f1')
from IPython.display import Image
Image('milestone/figures/sample_efficiency_venue_f1.png')

## 13. Ablations

In [ ]:
from pathlib import Path
from src.eval.ablations import run_ablations

run_ablations(Path('data/annotations/labels.csv'), Path('data/clips'),
              Path('data/results/ablations.csv'),
              ckpts={'both': Path('checkpoints/encoder.pt')},
              type_ckpt=Path('checkpoints/encoder.pt'),
              seeds=(0, 1, 2), group_by_venue=True, epochs=30, device='cuda')

## 14. Build figures

In [ ]:
!python -m src.viz.make_figures \
    --log checkpoints/train_log.csv --csv data/annotations/auto.csv \
    --emb data/embeddings.npz --labels data/annotations/labels.csv \
    --results data/results/sample_efficiency.csv --out milestone/figures

from pathlib import Path
import pandas as pd
from src.viz.tsne import plot_tsne
from src.viz.detection_examples import render_clip, _save_grid
from src.preprocess.detect import Box, HorseDetector

OUT = Path('milestone/figures'); OUT.mkdir(parents=True, exist_ok=True)
(OUT / 'det').mkdir(parents=True, exist_ok=True)

plot_tsne(Path('data/embeddings.npz'), Path('data/annotations/by_video.csv'),
          color_by='video', out_path=OUT / 'tsne_video.png',
          title='t-SNE colored by source video (venue clustering)')

detector = HorseDetector(weights='yolov8n.pt', device='cuda')
fdf = pd.read_csv('data/annotations/fences.csv').head(4)
rendered = []
for _, row in fdf.iterrows():
    clip = Path('data/clips') / f"{row['clip_id']}.mp4"
    if not clip.exists():
        continue
    fb = Box(float(row['x1']), float(row['y1']), float(row['x2']), float(row['y2']), label='fence')
    pole = int(row['pole_count']) if pd.notna(row.get('pole_count')) else None
    out_jpg = OUT / 'det' / f"{row['clip_id']}.jpg"
    render_clip(clip, fb, pole, detector, out_jpg)
    if out_jpg.exists():
        rendered.append(out_jpg)
_save_grid(rendered, OUT / 'det_grid.png', cols=2)

print('figures:')
!ls -la milestone/figures/*.png

### 14b. t-SNE by video and venue

In [ ]:
import pandas as pd
from pathlib import Path
from src.data.venues import VIDEO_VENUE
from src.viz.tsne import plot_tsne

OUT = Path('milestone/figures'); OUT.mkdir(parents=True, exist_ok=True)
bv = pd.read_csv('data/annotations/by_video.csv')
bv['venue'] = bv['video'].astype(str).map(VIDEO_VENUE)
bv.to_csv('data/annotations/by_venue.csv', index=False)

plot_tsne(Path('data/embeddings.npz'), Path('data/annotations/by_video.csv'),
          'video', OUT / 'tsne_video.png', 't-SNE by source video (6)')
plot_tsne(Path('data/embeddings.npz'), Path('data/annotations/by_venue.csv'),
          'venue',  OUT / 'tsne_venue.png',  't-SNE by venue (3)')

### 14c. Confusion matrix

In [ ]:
!python -m src.viz.confusion --labels data/annotations/labels.csv \
  --clips data/clips --ckpt checkpoints/encoder.pt --finetune \
  --out milestone/figures/confusion.png
from IPython.display import Image; Image("milestone/figures/confusion.png")

### 14d. DANN vs. info-only t-SNE

In [ ]:
!python -m src.ssl.train --clips data/clips --out checkpoints_info --epochs 20 --lambda-domain 0
!python -m src.ssl.embed --clips data/clips --ckpt checkpoints_info/encoder.pt --out data/embeddings_info.npz
!python -m src.viz.tsne --emb data/embeddings.npz --ann data/annotations/by_venue.csv --color-by venue --out milestone/figures/tsne_venue_adv.png --title "DANN encoder (venue)"
!python -m src.viz.tsne --emb data/embeddings_info.npz --ann data/annotations/by_venue.csv --color-by venue --out milestone/figures/tsne_venue_info.png --title "Info-only encoder (venue)"

## 15. Preview figures

In [ ]:
from IPython.display import Image, display
from pathlib import Path

for p in sorted(Path('milestone/figures').glob('*.png')):
    print(p.name)
    display(Image(filename=str(p)))

## 16. Export figures + results to Drive

In [ ]:
import shutil, glob
from pathlib import Path

EXPORT = Path('/content/drive/MyDrive/CS131/final_export')
(EXPORT / 'figures').mkdir(parents=True, exist_ok=True)
(EXPORT / 'results').mkdir(parents=True, exist_ok=True)

for src in glob.glob('milestone/figures/*.png'):
    shutil.copy(src, EXPORT / 'figures' / Path(src).name)
for src in glob.glob('data/results/*.csv'):
    shutil.copy(src, EXPORT / 'results' / Path(src).name)
if Path('checkpoints/train_log.csv').exists():
    shutil.copy('checkpoints/train_log.csv', EXPORT / 'results' / 'train_log.csv')

zip_path = shutil.make_archive('/content/drive/MyDrive/CS131/final_export_bundle',
                               'zip', EXPORT)
print('zipped ->', zip_path)
print('\nfigures:'); [print(' ', Path(p).name) for p in sorted(glob.glob(str(EXPORT/'figures/*.png')))]
print('\nresults:'); [print(' ', Path(p).name) for p in sorted(glob.glob(str(EXPORT/'results/*')))]

### 16b. Download bundle to local machine

In [ ]:
import shutil
from google.colab import files

local_zip = shutil.make_archive('/content/final_export_bundle', 'zip', EXPORT)
print('built:', local_zip)
files.download(local_zip)

## 17. Test A — leave-one-venue-out (honest cross-venue)

In [ ]:
from pathlib import Path
from src.eval.cross_venue import run_lovo, plot

labels = Path('data/annotations/labels.csv')
clips = Path('data/clips')
ckpt = Path('checkpoints/encoder.pt')

df_A = run_lovo(labels, clips, ckpt, level='venue',
                finetune=True, epochs=30, device='cuda', seeds=(0, 1, 2))
df_A.to_csv('data/results/cross_venue_A.csv', index=False)
plot(df_A, Path('milestone/figures/cross_venue_A.png'),
     'Test A: leave-one-venue-out (cross-venue)')
df_A.groupby('holdout')[['n_val','accuracy','macro_f1']].mean()

## 18. Test B — within-Wellington cross-competition

In [ ]:
df_B = run_lovo(labels, clips, ckpt, level='video', restrict_venue='wellington',
                finetune=True, epochs=30, device='cuda', seeds=(0, 1, 2))
df_B.to_csv('data/results/cross_venue_B.csv', index=False)
plot(df_B, Path('milestone/figures/cross_venue_B.png'),
     'Test B: leave-one-video-out within Wellington (cross-competition)')

print(f"Test A mean macro-F1 (cross-venue):        {df_A['macro_f1'].mean():.3f}")
print(f"Test B mean macro-F1 (cross-competition):  {df_B['macro_f1'].mean():.3f}")
print('B > A by a margin => venue-backdrop shortcut.')